### MiddleWare
Middleware provides a way to more tightly controls what happen inside the agent. Middleware is useful for follwing:
###### Tracking agent behaviour with logging, analytics, and debugging.
###### Transforming prompt, tools section and output formatting.
###### Adding retries , fallback and early termination logic.
###### Applying rate limit , guardrails and PII detection:
###### Guardrails are automated safety boundaries built into AI systems to ensure interactions remain safe, ethical, and within established policies. PII Detection is a specific guardrail function that automatically identifies, redacts, or masks sensitive personal data to prevent privacy leaks and ensure regulatory compliance

### Summerization Middlware:
##### Automatically summerize conversation history when approaching token limit, preserving recent messages while compressing older context. useful for the following:
- Long- running conversation that that exceeds context window
- Multi-turn dialog with extensive history
- Applications where preserving full conversation context matter

In [2]:
import os
from langchain.chat_models import init_chat_model
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')

model = init_chat_model('groq:qwen/qwen3-32b')
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001A68BDE6A50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A68BDE74D0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

# Message Based summerization

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),# it will store the memory in harddrive
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages",10),# message lenth reach to 10 summerization will trigger
            keep=("messages",4)
        )
    ]
)

In [5]:
# Run with thread id

config = {"configurable":{"thread_id":"test-1"}}

In [6]:
# questions
question = [
    "What is 2+2",
    "What is 10*2",
    "What is 3-2",
    "What is 2*4",
    "What is 300/3",
    "What is 4*2"
]
for q in question:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config=config)
    print("messages",response)
    print("messages lenght",len(response['messages']))

messages {'messages': [HumanMessage(content='What is 2+2', additional_kwargs={}, response_metadata={}, id='54bcce4f-986e-4cfc-99dd-25b9a3e2a338'), AIMessage(content='<think>\nOkay, so I need to figure out what 2 plus 2 is. Let me start by recalling basic addition. When you add two numbers together, you\'re combining their quantities. So, 2 is a number that represents two units, and adding another 2 would mean combining those two units with another two units.\n\nLet me visualize this. If I have two apples and I get two more apples, how many apples do I have in total? Starting with two, adding two more would make four. That seems straightforward. But wait, maybe I should double-check. Maybe I can use my fingers to count. Let\'s see, two fingers on one hand, and two on the other. If I put them together, that\'s four fingers. Yeah, that makes sense.\n\nAnother way to think about it is using number lines. If I start at 2 on a number line and then move 2 units to the right, where do I land? 

### Based on Token Size

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotel(city:str)->str:
    """Search Hotels- returns long response to use more token."""
    return f"""Hotel city:{city}:
    1.Grand hotel - 5 star ,spa"
    2. City in -4 star , business center 
    3. Budget Stay - 3 star, free wifi """

agent2 = create_agent(
    model=model,
    tools=[search_hotel],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens",550),
            keep = ("tokens",200)
        )
    ]
)
config ={"configurable":{"thread_id":"test-1"}}

# token counter (approximate)
def count_token(messages):
    totalchars = sum(len(str(m.content)) for m in messages)
    return totalchars//4 # 4 chars = 1 token

In [9]:
# run test
cities = ["paris","London","Tokyo","New-York","Dubai","Sigapore"]

for city in cities:
    response=agent2.invoke(
        {"messages":[HumanMessage(content=f"find hotels in {city}")]},config=config
    )

    tokens = count_token(response['messages'])
    print(f"{city} : - {tokens} tokens, {len(response['messages'])} messages")
    print("messages",response['messages'])

paris : - 141 tokens, 4 messages
messages [HumanMessage(content='find hotels in paris', additional_kwargs={}, response_metadata={}, id='4760feaa-6fb5-454d-9fce-046999ad767b'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to find hotels in Paris. Let me check the available tools. There\'s a function called search_hotel that takes a city parameter. Since the user mentioned Paris, I need to call this function with "Paris" as the city argument. I\'ll make sure to format the tool call correctly in JSON inside the XML tags.\n', 'tool_calls': [{'id': 'cvcazcc93', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotel'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 155, 'total_tokens': 250, 'completion_time': 0.138987718, 'completion_tokens_details': {'reasoning_tokens': 70}, 'prompt_time': 0.008229503, 'prompt_tokens_details': None, 'queue_time': 0.653830344, 'total_time': 0.147217221}, 

#### Human in the Loop Middleware
Pause agent execution for human approval , editing or rejection of tool calls before they execute, human in the loop for following:
- High - stake operations requires human approval( database write , financial write)
- Long running conversation wher human guidance is important

In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id:str)->str:
    '''Mock function to read an email with its id'''
    return f"Email content for id:{email_id}"

def send_email_tool(recipient: str, subject: str, body: str)->str:
    '''Mock function to send an email'''
    return f"Email sent to {recipient} with an subject : {subject} contains body: {body}"

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    tools=[read_email_tool,send_email_tool],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False
            }
        )
    ]
)

In [11]:
config = {"configurable":{"thread_id":"test-approve"}}
# step 1 : Request
response = agent.invoke(
    {"messages":[HumanMessage(content="Send email to john@gmail.com with subject: Hello and body: How are you")]},
    config=config
)
response

{'messages': [HumanMessage(content='Send email to john@gmail.com with subject: Hello and body: How are you', additional_kwargs={}, response_metadata={}, id='c4864ad0-1c6f-490f-b2ef-fff9e79d3c02'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to send an email to john@gmail.com with the subject "Hello" and body "How are you". Let me check the available tools. There\'s the send_email_tool which requires recipient, subject, and body. I need to make sure all required parameters are included. The recipient is provided as john@gmail.com, subject is Hello, and body is How are you. All required fields are present. I\'ll structure the tool call with these parameters.\n', 'tool_calls': [{'id': 'k4656rqjj', 'function': {'arguments': '{"body":"How are you","recipient":"john@gmail.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 136, 'prompt_tokens': 249, 'total_tokens': 385, 

In [12]:
from langgraph.types import Command
# step 2 : Approve
if "__interrupt__" in response:
    print("Paused approving")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"approve"}
                ]
            }
        ),
        config=config
    )
    print("result:",result['messages'][-1].content)


Paused approving
result: The email has been successfully sent to **john@gmail.com** with the subject **"Hello"** and the body **"How are you"**. Let me know if you need further assistance!


In [13]:
# step 4 : Reject
config = {"configurable":{"thread_id":"test-reject"}}
# step 1 : Request
response = agent.invoke(
    {"messages":[HumanMessage(content="Send email to john@gmail.com with subject: Hello and body: How are you")]},
    config=config
)
response

{'messages': [HumanMessage(content='Send email to john@gmail.com with subject: Hello and body: How are you', additional_kwargs={}, response_metadata={}, id='848ccafd-0069-497b-aec6-19e9888f2a24'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to send an email to john@gmail.com with the subject "Hello" and body "How are you". Let me check the available tools. There\'s the send_email_tool which requires recipient, subject, and body. I need to make sure all required parameters are included. The recipient is provided as john@gmail.com, subject is Hello, and body is How are you. All required fields are present. I\'ll structure the tool call with these parameters. No need to use the read_email_tool here since the task is about sending, not reading. Just need to format the JSON correctly for the send_email_tool.\n', 'tool_calls': [{'id': 't44j67mpa', 'function': {'arguments': '{"body":"How are you","recipient":"john@gmail.com","subject":"Hello"}', 'name

In [14]:
from langgraph.types import Command
# step 2 : Approve
if "__interrupt__" in response:
    print("Paused approving")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"reject"}
                ]
            }
        ),
        config=config
    )
    print("result:",result['messages'][-1].content)


Paused approving
result: The email was not sent as the action was canceled. Would you like me to proceed with sending it or assist with anything else?


In [17]:
# step 5: edit
# step 4 : Reject
config = {"configurable":{"thread_id":"test-edit"}}
# step 1 : Request
response = agent.invoke(
    {"messages":[HumanMessage(content="Send email to johny@gmail.com with subject: Hello and body: How are you")]},
    config=config
)
response

{'messages': [HumanMessage(content='Send email to john@gmail.com with subject: Hello and body: How are you', additional_kwargs={}, response_metadata={}, id='b4e2abe8-7f1a-4501-9356-8dc89f4c08c5'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to send an email to john@gmail.com with the subject "Hello" and body "How are you". Let me check the available tools. There\'s the send_email_tool which requires recipient, subject, and body. I need to make sure all required parameters are included. The recipient is provided as john@gmail.com, subject is Hello, and body is How are you. All required fields are present. I\'ll call the send_email_tool with these parameters.\n', 'tool_calls': [{'id': 'jphbh7bna', 'function': {'arguments': '{"body":"How are you","recipient":"john@gmail.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 137, 'prompt_tokens': 249, 'total_tokens': 386,

In [18]:
from langgraph.types import Command
# step 2 : Approve
if "__interrupt__" in response:
    print("Paused approving")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {
                        "type":"reject",
                        "edited_action":{
                            "name":"send_email_tool",# tool name
                            "args":{
                                "recipient":"edited@gmail.com",
                                "subject":"edited subject",
                                "body":"This was edited by human before sending"
                            } 
                        }
                    }
                ]
            }
        ),
        config=config
    )
    print("result:",result['messages'][-1].content)


Paused approving
result: The tool call to send the email to `johny@gmail.com` was rejected. Would you like to:  

1. **Resubmit the request** for the same email?  
2. **Cancel** and stop further actions?  
3. **Edit the email details** (e.g., recipient, subject, or body) before resending?  

Let me know your choice!
